# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [10]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [11]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [ ]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

In [21]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [22]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii 5 ani, politica românească a fost marcată de alternanțe guvernamentale frecvente și de o consolidare a rolului Uniunii Europene în procesul decizional. De asemenea, s-au observat eforturi de combatere a corupției și o creștere a importanței dezbaterilor legate de statul de drept.

--- Gemini 2.5 Flash ---
Politica românească a fost redefinită de formarea unei coaliții guvernamentale largi între partide tradițional rivale (PSD și PNL), incluzând un acord de rotație a premierilor, pentru a asigura stabilitatea. Peisajul politic a cunoscut o fragmentare crescută după alegerile din 2020, concomitent cu eforturi de reformare a legislației justiției și implementarea Planului Național de Redresare și Reziliență.

--- OpenRouter Free ---
În ultimii 5 ani, România a introdus reforme majore în sistemul de sănătate, consolidând asigurările sociale și îmbunătățind accesul la servicii medicale. De asemenea, au fost lansate măsuri pentru stimularea creșterii

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [23]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---


KeyboardInterrupt: 

## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [24]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [25]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicieni', 'populism': True, 'explicatie_scurta': 'Comentariul exprimă o frustrare generală față de politicieni, acuzându-i de corupție și ignorarea voinței poporului, un discurs tipic populist.'}

--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicienii', 'populism': True, 'explicatie_scurta': "Comentariul exprimă o critică generalizată la adresa politicienilor, acuzându-i de corupție și de ignorarea poporului, folosind o retorică populistă prin contrastul 'oameni simpli' vs. 'politicieni'."}

--- OpenRouter Free ---
{'ton': 'negativ', 'emotie_dominanta': 'frica', 'tinta_principala': 'criticizare a sistemului politică', 'populism': True, 'explicatie_scurta': 'Comentariul expresează o frustrație mare față de corupție politică și o sentimente de impunitate, dar și de desconectare dintre politicieni și poporul. Suggerează o crisis de cre

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [26]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea de noi alegeri, ceea ce implică o perioadă de incertitudine politică și o posibilă reconfigurare a forțelor politice. Acest proces poate influența stabilitatea guvernamentală și direcția politicilor publice pe termen scurt și mediu.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională implică faptul că procesul electoral anterior a fost considerat neconform cu legea fundamentală. Aceasta necesită organizarea unor noi alegeri pentru a asigura legitimitatea reprezentării politice.

temperature=1.2:
Anularea alegerilor de către Curtea Constituțională înseamnă că rezultatul votului nu va fi recunoscut oficial și procesul electoral va trebui reluat. Acest lucru poate conduce la o perioadă de incertitudine politică și la reorganizarea calendarului electoral.

[ Gemini 2.5 Flash ]

temperature

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| OpenRouter Free | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| Llama / alt model testat | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
### Decizie
**Model principal ales:**  
**Model de rezervă:**  
**Temperature recomandată:**  
**De ce am ales acest model?**  
Scrieți 2-3 propoziții. Menționați calitatea răspunsului, stabilitatea și dacă modelul poate fi folosit pentru adnotarea comentariilor.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [17]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales